# Qwen3.8-27B on Kaggle for OpenCode (v7: llama.cpp + MTP)

Serve Qwen3.8-27B as an OpenAI-compatible API on Kaggle's 2×T4 GPUs. This version keeps the Q4_K_XL target model and enables speculative decoding with Qwen3.8's separate MTP draft GGUF. The notebook searches attached Kaggle inputs for both files and downloads either one if missing.

## Kaggle prerequisites

In Notebook Settings:
1. Accelerator: **GPU T4 × 2**
2. Internet: **On** (downloads llama.cpp, cloudflared, and missing model files)
3. Optionally attach a Kaggle dataset containing both `Qwen3.8-27B-UD-Q4_K_XL.gguf` and `mtp-Qwen3.8-27B-Q4_0.gguf`. The notebook searches `/kaggle/input` automatically.

If the target model or MTP draft is missing, the notebook downloads it from Unsloth. The MTP draft is about 1.37 GB and needs additional GPU memory. The active target context remains 131K; if model loading runs out of VRAM after adding MTP, lower `MAX_MODEL_LEN` (for example to 65,536) and rerun from the model configuration cell.

In [3]:
# Cell 1 — Verify that this is the intended Kaggle runtime.
import os, platform, subprocess, sys

def run_text(cmd):
    return subprocess.run(cmd, shell=True, text=True, capture_output=True).stdout.strip()

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('KAGGLE_KERNEL_RUN_TYPE:', os.environ.get('KAGGLE_KERNEL_RUN|_TYPE', '<not set>'))
print(run_text('nvidia-smi --query-gpu=index,name,compute_cap,memory.total --format=csv,noheader'))

import torch
assert torch.cuda.is_available(), 'CUDA is unavailable. Enable a GPU accelerator in Kaggle Notebook Settings.'
gpu_count = torch.cuda.device_count()
assert gpu_count >= 2, f'Expected 2 GPUs, found {gpu_count}. Select a 2×T4 accelerator.'
for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {props.name}; compute capability {props.major}.{props.minor}; VRAM {props.total_memory / 2**30:.2f} GiB')
    assert (props.major, props.minor) == (7, 5), 'This notebook is tuned for Turing sm_75/T4 GPUs.'
print('Runtime checks passed. Two T4-class GPUs are visible.')

Python: 3.13.15
Platform: Linux-6.18.48+-x86_64-with-glibc2.39
KAGGLE_KERNEL_RUN_TYPE: <not set>
0, Tesla T4, 7.5, 15360 MiB
1, Tesla T4, 7.5, 15360 MiB
GPU 0: Tesla T4; compute capability 7.5; VRAM 14.56 GiB
GPU 1: Tesla T4; compute capability 7.5; VRAM 14.56 GiB
Runtime checks passed. Two T4-class GPUs are visible.


In [2]:
# Cell 2 — Download llama.cpp CUDA server and its matching runtime libraries.
# Upstream publishes the server binary and CUDA runtime as separate release assets.
import os, json, tarfile, urllib.request, subprocess, re
from pathlib import Path

LLAMA_DIR = Path('/kaggle/working/llama.cpp')
LLAMA_DIR.mkdir(parents=True, exist_ok=True)
headers = {'Accept': 'application/vnd.github+json', 'User-Agent': 'kaggle-notebook'}

def is_linux_cuda12_x64_archive(name):
    name = name.lower()
    is_linux = 'ubuntu' in name or 'linux' in name
    is_cuda12 = bool(re.search(r'cuda[-_. ]?12(?:[.-]|$)', name))
    is_x64 = 'x64' in name or 'x86_64' in name
    is_archive = name.endswith(('.tar.gz', '.tgz', '.tar.xz'))
    return is_linux and is_cuda12 and is_x64 and is_archive

request = urllib.request.Request(
    'https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=30',
    headers=headers
)
with urllib.request.urlopen(request, timeout=30) as response:
    releases = json.load(response)

selected_release = None
server_asset = None
runtime_asset = None
recent_asset_names = []
for candidate_release in releases:
    candidate_assets = candidate_release.get('assets', [])
    recent_asset_names.extend(
        f"{candidate_release.get('tag_name')}: {a.get('name', '<unnamed>')}"
        for a in candidate_assets[:12]
    )
    server_matches = [
        a for a in candidate_assets
        if a.get('name', '').lower().startswith('llama-')
        and '-bin-' in a.get('name', '').lower()
        and 'cudart' not in a.get('name', '').lower()
        and is_linux_cuda12_x64_archive(a.get('name', ''))
    ]
    if server_matches:
        selected_release = candidate_release
        server_asset = next((a for a in server_matches if '12.8' in a['name']), server_matches[0])
        break

if server_asset is None:
    available = '\n'.join(recent_asset_names[:80]) or 'No release assets returned by GitHub.'
    raise RuntimeError(
        'No llama.cpp Ubuntu x64 CUDA 12 server package found in the 30 most recent releases. '
        f'Available release assets:\n{available}'
    )

runtime_matches = [
    a for a in selected_release.get('assets', [])
    if a.get('name', '').lower().startswith('cudart-llama-')
    and is_linux_cuda12_x64_archive(a.get('name', ''))
]
runtime_asset = next((a for a in runtime_matches if '12.8' in a['name']), runtime_matches[0] if runtime_matches else None)

def download_and_extract(asset):
    archive = Path('/kaggle/working') / asset['name']
    expected_size = asset.get('size')
    if not archive.exists() or (expected_size and archive.stat().st_size != expected_size):
        print('Downloading:', asset['name'])
        urllib.request.urlretrieve(asset['browser_download_url'], archive)
    mode = 'r:xz' if asset['name'].lower().endswith('.tar.xz') else 'r:gz'
    with tarfile.open(archive, mode) as package:
        package.extractall(LLAMA_DIR, filter='data')
    return archive

server_archive = download_and_extract(server_asset)
if runtime_asset:
    download_and_extract(runtime_asset)
    print('CUDA runtime package:', runtime_asset['name'])
else:
    print('No separate CUDA runtime archive listed; using libraries available in the Kaggle image.')

server_candidates = list(LLAMA_DIR.rglob('llama-server'))
if not server_candidates:
    raise FileNotFoundError(f'Server package {server_asset["name"]} contained no llama-server: {server_archive}')
LLAMA_SERVER = str(server_candidates[0])
os.chmod(LLAMA_SERVER, 0o755)

# Release packages place shared libraries beside the executable and/or in nested lib directories.
# Add every extracted shared-library directory, not only the CUDA runtime directory.
shared_lib_dirs = {str(Path(LLAMA_SERVER).parent)}
shared_lib_dirs.update(str(p.parent) for p in LLAMA_DIR.rglob('*.so*'))
previous_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
ld_parts = sorted(shared_lib_dirs)
if previous_ld_path:
    ld_parts.append(previous_ld_path)
os.environ['LD_LIBRARY_PATH'] = os.pathsep.join(ld_parts)

print('llama.cpp release:', selected_release.get('tag_name'))
print('Selected server package:', server_asset['name'])
print('Server binary:', LLAMA_SERVER)
print('Shared-library search directories:', len(shared_lib_dirs))

# Diagnose missing shared libraries before invoking --version.
ldd_result = subprocess.run(['ldd', LLAMA_SERVER], text=True, capture_output=True, env=os.environ.copy())
ldd_text = ldd_result.stdout + ldd_result.stderr
missing_libraries = [line.strip() for line in ldd_text.splitlines() if 'not found' in line]
if missing_libraries:
    raise RuntimeError(
        'llama-server still has unresolved shared libraries after adding extracted library paths:\n'
        + '\n'.join(missing_libraries)
        + '\nDependency report:\n' + ldd_text[-5000:]
    )

version = subprocess.run([LLAMA_SERVER, '--version'], text=True, capture_output=True, env=os.environ.copy())
if version.returncode:
    raise RuntimeError(
        'llama-server could not run after shared-library paths were configured. '
        f'Exit code: {version.returncode}\n{version.stderr[-5000:]}'
    )
print(version.stdout.strip())

device_result = subprocess.run([LLAMA_SERVER, '--list-devices'], text=True, capture_output=True, env=os.environ.copy())
device_text = device_result.stdout + device_result.stderr
print('llama.cpp devices:\n', device_text)
if 'CUDA0' not in device_text or 'CUDA1' not in device_text:
    raise RuntimeError('llama.cpp must detect both CUDA0 and CUDA1 for the dual-GPU configuration.')

CUDA runtime package: cudart-llama-b11719-bin-ubuntu-cuda-12.8-x64.tar.gz
llama.cpp release: b11719
Selected server package: llama-b11719-bin-ubuntu-cuda-12.8-x64.tar.gz
Server binary: /kaggle/working/llama.cpp/llama-b11718/llama-server
Shared-library search directories: 6

llama.cpp devices:
 Available devices:
  CUDA0: Tesla T4 (14911 MiB, 14806 MiB free)
  CUDA1: Tesla T4 (14911 MiB, 14806 MiB free)



## Install llama.cpp

This notebook downloads a prebuilt Ubuntu x64 CUDA 12 llama.cpp server and its matching runtime libraries. It selects a recent release and checks that the server advertises speculative MTP support before continuing.

The target Q4 GGUF and the separate MTP draft are loaded by llama-server. MTP can improve decode throughput when draft tokens are accepted, but the gain depends on the prompt and sampling settings. The draft also uses additional VRAM.

In [11]:
# Cell 3 — Select/download the target and MTP draft, then configure the endpoint.
import os, secrets, json, time, signal, subprocess, urllib.request, urllib.error, glob, shutil, sys
from pathlib import Path

GGUF_REPO = 'unsloth/Qwen3.8-27B-GGUF'
GGUF_FILENAME = 'Qwen3.8-27B-UD-Q4_K_XL.gguf'
MTP_FILENAME = 'MTP/mtp-Qwen3.8-27B-Q4_0.gguf'
download_dir = Path('/tmp/qwen3.8-27b-gguf')
download_dir.mkdir(parents=True, exist_ok=True)

def get_hf_download():
    try:
        from huggingface_hub import hf_hub_download
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
        from huggingface_hub import hf_hub_download
    return hf_hub_download

hf_download = None

# Select the target GGUF from an explicit path, an attached dataset, or Hugging Face.
configured_path = os.environ.get('KAGGLE_GGUF_PATH', '').strip()
if configured_path:
    MODEL_PATH = Path(configured_path)
else:
    matches = glob.glob('/kaggle/input/**/*.gguf', recursive=True)
    preferred = [Path(p) for p in matches if GGUF_FILENAME.lower() in Path(p).name.lower()]
    matches_q4 = [Path(p) for p in matches if 'qwen3.8-27b' in Path(p).name.lower() and ('q4' in Path(p).name.lower() or 'iq4' in Path(p).name.lower())]
    MODEL_PATH = (preferred or matches_q4 or [None])[0]

if MODEL_PATH is None:
    print(f'Downloading target model: {GGUF_REPO}/{GGUF_FILENAME}')
    hf_download = get_hf_download()
    MODEL_PATH = Path(hf_download(repo_id=GGUF_REPO, filename=GGUF_FILENAME, local_dir=str(download_dir)))

if not MODEL_PATH.is_file() or MODEL_PATH.suffix.lower() != '.gguf':
    raise FileNotFoundError(f'KAGGLE_GGUF_PATH must point to one GGUF file: {MODEL_PATH}')

# Select/download the separate Qwen3.8 MTP draft GGUF.
configured_mtp_path = os.environ.get('KAGGLE_MTP_PATH', '').strip()
if configured_mtp_path:
    MTP_PATH = Path(configured_mtp_path)
else:
    mtp_matches = glob.glob('/kaggle/input/**/mtp-Qwen3.8-27B-Q4_0.gguf', recursive=True)
    MTP_PATH = Path(mtp_matches[0]) if mtp_matches else None

if MTP_PATH is None:
    print(f'Downloading MTP draft: {GGUF_REPO}/{MTP_FILENAME} (~1.37 GB)')
    if hf_download is None:
        hf_download = get_hf_download()
    MTP_PATH = Path(hf_download(repo_id=GGUF_REPO, filename=MTP_FILENAME, local_dir=str(download_dir)))

if not MTP_PATH.is_file() or MTP_PATH.suffix.lower() != '.gguf':
    raise FileNotFoundError(f'KAGGLE_MTP_PATH must point to the MTP draft GGUF: {MTP_PATH}')

# Fail early with a useful message if this llama.cpp build lacks the MTP options.
help_result = subprocess.run([LLAMA_SERVER, '--help'], capture_output=True, text=True, env=os.environ.copy(), timeout=30)
llama_help = help_result.stdout + help_result.stderr
if 'draft-mtp' not in llama_help or ('--model-draft' not in llama_help and '--spec-draft-model' not in llama_help):
    raise RuntimeError('The selected llama.cpp build does not advertise MTP speculative decoding. Use a newer llama.cpp release.')

MODEL_ID = 'qwen3.8-27b'
PORT = 8000
MAX_MODEL_LEN = 131_072
N_GPU_LAYERS = 64  # Qwen3.8-27B has 64 transformer layers.
MAX_OUTPUT_TOKENS = 16_384
MTP_DRAFT_TOKENS = 3  # Compare 2 vs 3 with the throughput benchmark.

Path('/kaggle/working').mkdir(parents=True, exist_ok=True)
key_file = Path('/kaggle/working/api_key.txt')
if key_file.exists():
    api_key = key_file.read_text().strip()
else:
    api_key = 'kgl-' + secrets.token_urlsafe(32)
    key_file.write_text(api_key)
os.environ['OPENCODE_API_KEY'] = api_key
print({
    'model_file': str(MODEL_PATH),
    'mtp_draft_file': str(MTP_PATH),
    'mtp_draft_tokens': MTP_DRAFT_TOKENS,
    'port': PORT,
    'context': MAX_MODEL_LEN,
    'gpu_layers': N_GPU_LAYERS,
    'split_mode': 'layer',
    'max_output_tokens': MAX_OUTPUT_TOKENS,
})
print('Session API Key:', api_key)

{'model_file': '/tmp/qwen3.8-27b-gguf/Qwen3.8-27B-UD-Q4_K_XL.gguf', 'mtp_draft_file': '/tmp/qwen3.8-27b-gguf/MTP/mtp-Qwen3.8-27B-Q4_0.gguf', 'mtp_draft_tokens': 3, 'port': 8000, 'context': 131072, 'gpu_layers': 64, 'split_mode': 'layer', 'max_output_tokens': 16384}
Session API Key: kgl-wfabn_-Avsw-Ze7lUTS2B23hpGtfnTJzQGHTyy3nIPY


## Start llama.cpp with both GPUs and MTP

The server uses the Q4_K_XL target model plus the separate Qwen3.8 MTP draft GGUF. MTP speculative decoding proposes up to `MTP_DRAFT_TOKENS` tokens for the target to verify. It can improve decode throughput, but the gain depends on acceptance rate and it uses extra GPU memory.

- `--device CUDA0,CUDA1`: make both T4s available.
- `--split-mode layer --tensor-split 1,1`: distribute target layers between the two GPUs. This notebook retains layer split because its tested CUDA build reported unsupported split buffers for row/tensor splitting.
- `--ctx-size 131072 --parallel 1`: keep the configured 131K context for one request slot.
- `--model-draft MTP_PATH --spec-type draft-mtp`: load the MTP sidecar and enable speculative decoding.
- `--spec-draft-n-max 2`: start conservatively; compare 2 and 3 using the server-side throughput cell.

The MTP sidecar is about 1.37 GB. If model loading fails with CUDA out-of-memory, lower `MAX_MODEL_LEN` to 65,536 first. Confirm in the server log that the draft model loaded and MTP speculation initialized.

In [12]:
# Cell 4 — Launch llama-server with both T4 GPUs and MTP speculative decoding.
subprocess.run(['pkill', '-9', '-f', 'llama-server'], check=False)
time.sleep(1)

print('GPU memory before model load:')
subprocess.run([
    'nvidia-smi',
    '--query-gpu=index,name,memory.used,memory.free',
    '--format=csv,noheader'
], check=False)

log_path = '/kaggle/working/llama-server.log'
log_file = open(log_path, 'w')
llama_args = [
    LLAMA_SERVER,
    '--model', str(MODEL_PATH),
    '--model-draft', str(MTP_PATH),
    '--spec-type', 'draft-mtp',
    '--spec-draft-n-max', str(MTP_DRAFT_TOKENS),
    '--alias', MODEL_ID,
    '--host', '0.0.0.0',
    '--port', str(PORT),
    '--api-key', api_key,
    '--ctx-size', str(MAX_MODEL_LEN),
    '--device', 'CUDA0,CUDA1',
    '--n-gpu-layers', str(N_GPU_LAYERS),
    '--split-mode', 'layer',
    '--tensor-split', '1,1',
    '--parallel', '1',
    '--batch-size', '256',
    '--ubatch-size', '64',
    '--flash-attn', 'on',
    '--jinja',
    '--no-webui',
    '--cache-type-k', 'q8_0',
    '--cache-type-v', 'q8_0',
]
llama_proc = subprocess.Popen(
    llama_args, stdout=log_file, stderr=subprocess.STDOUT,
    env=os.environ.copy(), start_new_session=True
)
print('llama-server PID:', llama_proc.pid)
print('Log:', log_path)
print(f'CUDA0 and CUDA1 selected; {N_GPU_LAYERS} target layers split 1:1; context={MAX_MODEL_LEN}; MTP draft tokens={MTP_DRAFT_TOKENS}.')

GPU memory before model load:
0, Tesla T4, 3 MiB, 14909 MiB
1, Tesla T4, 3 MiB, 14909 MiB
llama-server PID: 369
Log: /kaggle/working/llama-server.log
CUDA0 and CUDA1 selected; 64 target layers split 1:1; context=131072; MTP draft tokens=3.


## Start a Cloudflare Quick Tunnel

This starts while llama.cpp loads so the HTTPS URL is available sooner. Inference requests work after the server finishes loading. The URL is temporary. Keep the URL and API key private.

In [13]:
# Cell 5 — Download cloudflared and expose localhost:8000 through HTTPS.
cloudflared_path = '/kaggle/working/cloudflared'
if not Path(cloudflared_path).exists() or Path(cloudflared_path).stat().st_size < 1_000_000:
    subprocess.run([
        'wget', '-q', '--show-progress', '-O', cloudflared_path,
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    ], check=True)
# Kaggle may clear executable bits, including on a file left by an earlier cell run.
os.chmod(cloudflared_path, 0o755)

tunnel_log_path = '/kaggle/working/cloudflared.log'
tunnel_log = open(tunnel_log_path, 'w')
cloudflared_proc = subprocess.Popen(
    [cloudflared_path, 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{PORT}'],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
    env=os.environ.copy(), start_new_session=True
)

tunnel_url = None
for _ in range(30):
    time.sleep(2)
    text = Path(tunnel_log_path).read_text(errors='replace')
    import re
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
    if match:
        tunnel_url = match.group(0)
        break

if not tunnel_url:
    print(Path(tunnel_log_path).read_text(errors='replace')[-12000:])
    raise RuntimeError('Cloudflare Tunnel URL was not found. Inspect cloudflared.log.')

public_base_url = tunnel_url + '/v1'
print('Public base URL:', public_base_url)
print('Keep this URL and the API key private.')

Public base URL: https://quantitative-victor-luggage-tiles.trycloudflare.com/v1
Keep this URL and the API key private.


In [14]:
# Cell 6 — Wait for llama.cpp and print the advertised model ID.
def get_json(url):
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {api_key}'})
    with urllib.request.urlopen(req, timeout=10) as response:
        return json.loads(response.read().decode())

ready = False
for attempt in range(180):  # up to 9 minutes
    try:
        models = get_json(f'http://127.0.0.1:{PORT}/v1/models')
        ready = True
        break
    except Exception:
        time.sleep(3)
        if attempt % 10 == 0:
            print(f'Waiting for llama.cpp... {attempt * 3}s')
        if llama_proc.poll() is not None:
            break

if not ready:
    print(Path(log_path).read_text(errors='replace')[-12000:])
    raise RuntimeError('llama-server did not become ready. Inspect /kaggle/working/llama-server.log.')

served_model_id = models['data'][0]['id']
print('llama.cpp is ready.')
print('Served model ID:', served_model_id)
print('OpenAI endpoint:', f'http://127.0.0.1:{PORT}/v1')

Waiting for llama.cpp... 0s
llama.cpp is ready.
Served model ID: qwen3.8-27b
OpenAI endpoint: http://127.0.0.1:8000/v1


## Generate OpenCode configuration

The generated file follows OpenCode's documented `@ai-sdk/openai-compatible` provider format. Copy it to the OpenCode project directory, or use the displayed values in your local OpenCode configuration.

In [16]:
# Cell 7 — Write the OpenCode config and print the provider block for copying.
provider_entry = {
    'npm': '@ai-sdk/openai-compatible',
    'name': 'Kaggle Qwen3.8-27B',
    'options': {
        'baseURL': public_base_url,
        'apiKey': api_key
    },
    'models': {
        served_model_id: {
            'name': 'Qwen3.8-27B GGUF',
            'limit': {'context': MAX_MODEL_LEN, 'output': MAX_OUTPUT_TOKENS}
        }
    }
}

opencode_config = {
    '$schema': 'https://opencode.ai/config.json',
    'provider': {
        'kaggle-qwen': provider_entry
    }
}
config_path = Path('/kaggle/working/opencode.jsonc')
config_path.write_text(json.dumps(opencode_config, indent=2) + '\n')

# Print this entry inside the existing "provider" object in your OpenCode config.
provider_lines = json.dumps({'kaggle-qwen': provider_entry}, indent=2).splitlines()
provider_snippet = '\n'.join('  ' + line for line in provider_lines[1:-1]) + ','
# print('OpenCode provider block (paste inside "provider": { ... }):')
print(provider_snippet)
print()
print('Full config saved to:', config_path)

    "kaggle-qwen": {
      "npm": "@ai-sdk/openai-compatible",
      "name": "Kaggle Qwen3.8-27B",
      "options": {
        "baseURL": "https://quantitative-victor-luggage-tiles.trycloudflare.com/v1",
        "apiKey": "kgl-wfabn_-Avsw-Ze7lUTS2B23hpGtfnTJzQGHTyy3nIPY"
      },
      "models": {
        "qwen3.8-27b": {
          "name": "Qwen3.8-27B GGUF",
          "limit": {
            "context": 131072,
            "output": 16384
          }
        }
      }
    },

Full config saved to: /kaggle/working/opencode.jsonc


In [18]:
# Cell 8 — Measure llama.cpp server-side prompt and generation throughput.
# This uses llama.cpp's timings in the API response, not client-side token timing.
import json
import time
import urllib.request
import urllib.error

benchmark_messages = [
    {
        "role": "user",
        "content": (
            "Write a detailed explanation of how a thread-safe priority queue works. "
            "Use clear prose and enough detail to generate a few hundred tokens."
        ),
    }
]
benchmark_request = {
    "model": served_model_id,
    "messages": benchmark_messages,
    "temperature": 0,
    "max_tokens": 5120,
    "stream": False,
    "timings_per_token": True,
}
request = urllib.request.Request(
    f"http://127.0.0.1:{PORT}/v1/chat/completions",
    data=json.dumps(benchmark_request).encode("utf-8"),
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    },
    method="POST",
)

started = time.perf_counter()
try:
    with urllib.request.urlopen(request, timeout=900) as response:
        benchmark_result = json.loads(response.read().decode("utf-8"))
except urllib.error.HTTPError as exc:
    print(exc.read().decode("utf-8", errors="replace"))
    raise

client_elapsed_s = time.perf_counter() - started
server_timings = benchmark_result.get("timings")
if not isinstance(server_timings, dict):
    raise RuntimeError(
        "llama.cpp response did not include server timings. "
        "Check the server log/version and ensure timings_per_token is supported."
    )

prompt_n = server_timings.get("prompt_n")
prompt_ms = server_timings.get("prompt_ms")
prompt_tps = server_timings.get("prompt_per_second")
predicted_n = server_timings.get("predicted_n")
predicted_ms = server_timings.get("predicted_ms")
generation_tps = server_timings.get("predicted_per_second")
usage = benchmark_result.get("usage", {})

print("Server-side llama.cpp throughput")
print(f"Prompt tokens:       {prompt_n if prompt_n is not None else usage.get('prompt_tokens', 'n/a')}")
print(f"Prompt processing:   {prompt_ms:.1f} ms" if isinstance(prompt_ms, (int, float)) else "Prompt processing:   n/a")
print(f"Prompt throughput:   {prompt_tps:.2f} tokens/s" if isinstance(prompt_tps, (int, float)) else "Prompt throughput:   n/a")
print(f"Generated tokens:    {predicted_n if predicted_n is not None else usage.get('completion_tokens', 'n/a')}")
print(f"Generation time:     {predicted_ms:.1f} ms" if isinstance(predicted_ms, (int, float)) else "Generation time:     n/a")
print(f"Generation speed:    {generation_tps:.2f} tokens/s" if isinstance(generation_tps, (int, float)) else "Generation speed:    n/a")
print(f"Client wall time:    {client_elapsed_s:.2f} s (includes request/network overhead)")
print(f"Finish reason:       {benchmark_result.get('choices', [{}])[0].get('finish_reason', 'n/a')}")


Server-side llama.cpp throughput
Prompt tokens:       68
Prompt processing:   1636.7 ms
Prompt throughput:   41.55 tokens/s
Generated tokens:    820
Generation time:     102148.8 ms
Generation speed:    8.02 tokens/s
Client wall time:    106.35 s (includes request/network overhead)
Finish reason:       stop


## Local OpenCode setup

Copy /kaggle/working/opencode.jsonc (or the JSON snippet printed by Cell 7) into your local OpenCode configuration.

The generated config contains the session API key and temporary Cloudflare URL. In OpenCode, select the provider kaggle-qwen and model ID qwen3.8-27b. llama.cpp supplies the OpenAI-compatible chat endpoint.

In [17]:
# # Cell 9 — Optional cleanup when finished.
# # Run manually to stop the tunnel and llama.cpp server.
# for process_name in ('cloudflared_proc', 'llama_proc'):
#     process = globals().get(process_name)
#     if process is not None and process.poll() is None:
#         process.terminate()
#         print(f'Stop requested for {process_name}.')
# print('Kaggle may retain GPU allocation until the session is stopped.')

## Support and limitations

- Use a Kaggle 2×T4 accelerator with Internet enabled.
- The notebook uses the Q4_K_XL target GGUF and the separate Qwen3.8 MTP draft GGUF. It searches attached datasets first and downloads missing files to `/tmp`.
- The configured target context is 131,072 tokens and output limit is 16,384. The MTP draft adds about 1.37 GB of model data and may require lowering context if startup runs out of VRAM.
- The target has 64 transformer layers; `N_GPU_LAYERS=64`. Check llama-server's startup log to confirm actual CUDA offload and that the MTP draft initialized.
- MTP speculative decoding may improve generation speed, but the gain depends on draft acceptance, prompt, and sampling settings. Compare identical requests with 2 and 3 draft tokens using the benchmark cell.
- This notebook downloads a prebuilt llama.cpp CUDA 12 server and its matching runtime library package; it does not compile llama.cpp in the Kaggle session.
- GGUF is served by llama.cpp here; this endpoint is OpenAI-compatible but does not use vLLM.
- Cloudflare Quick Tunnel is temporary. Anyone with the URL and API key can access the endpoint, so keep both private.
- Public Kaggle datasets mount read-only under `/kaggle/input`; downloaded files under `/tmp` disappear when the session ends.

Sources: [llama.cpp speculative decoding](https://github.com/ggml-org/llama.cpp/blob/master/docs/speculative.md), [llama.cpp multi-GPU options](https://github.com/ggml-org/llama.cpp/blob/master/tools/server/README.md), [Qwen3.8 target and MTP files](https://huggingface.co/unsloth/Qwen3.8-27B-GGUF/tree/main).